In [24]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [25]:
from pygraphon.graphons.StepGraphon import StepGraphon
import numpy as np
from pygraphon.plots.plot_matrices import spy
import matplotlib.pyplot as plt
from pygraphon.utils.utils_graph import edge_density
from tqdm import tqdm
from pygraphon.plots.plots_graphon import plot_sample

from SmoothNetHist import SmoothNetHist
from experiment_smoothing import smooth_hist, initialisation_hist, smoothing_histLC, flat_to_tensor

from gen_sbm import AssociativeFullyRandom

In [32]:
#recall parameters of experiment
n=5000
k=10
theta_intra_min, theta_intra_max = 0.4, 0.6
theta_inter_min, theta_inter_max = 0.01, 0.2
theta = AssociativeFullyRandom(
                K=k,
                min_inside_classes=theta_intra_min,
                max_inside_classes=theta_intra_max,
                min_outside_classes=theta_inter_min,
                max_outside_classes=theta_inter_max,
            ).Theta
graphon_mat = StepGraphon(theta, 1/k)
# Set Adjacency
A = graphon_mat.draw(rho=1, n=n, exchangeable=False)

In [27]:
from pygraphon.estimators.networkhistogram.HistogramEstimator import HistogramEstimator

my_hist = HistogramEstimator()
my_hist.fit(graph= A, bandwidthHist= None)


Log likelihood: -2.6161:   1%|          | 5/500 [00:08<13:58,  1.69s/it]


In [28]:
my_hist.get_graphon().graphon

array([[4.3743687 , 3.89408507, 1.0501659 , 0.67366596, 1.3970837 ,
        0.93721592, 1.16849445, 1.11874267, 1.21017837, 1.32985157,
        0.60643383, 0.30523388, 0.32002495, 0.40608208, 0.45583385,
        0.71534988, 0.74358738, 0.39666958, 0.54323562, 0.94528377,
        0.850352  ],
       [3.89408507, 3.82620903, 1.19135338, 0.91704628, 1.28951229,
        0.90091057, 1.15639267, 1.1416016 , 1.15908195, 1.23976051,
        0.68038917, 0.410116  , 0.4195285 , 0.5015517 , 0.46255707,
        0.81754272, 0.63870525, 0.44104278, 0.55937134, 0.857882  ,
        0.79064987],
       [1.0501659 , 1.19135338, 3.59440021, 3.99493327, 1.22496944,
        0.876707  , 0.75972309, 0.91166771, 0.73014095, 0.68711239,
        0.839057  , 0.83233379, 0.71669452, 0.76779094, 0.59164276,
        0.7852713 , 0.75837845, 0.41549457, 0.47869278, 0.4571785 ,
        0.54700062],
       [0.67366596, 0.91704628, 3.99493327, 4.37164153, 1.2047998 ,
        0.90359985, 0.78661594, 0.88343021, 0.6655981

In [33]:
from SmoothNetHist import SmoothNetHist
smooth_test = SmoothNetHist()
test_return_first, assignments = smooth_test._first_approximate_graphon_from_adjacency(A)

Log likelihood: -2.5216:   1%|          | 5/500 [00:19<31:42,  3.84s/it]


In [30]:
test_return_first

# Testing if the smoothing along the tensor are the same

In [34]:
scaled = test_return_first.graphon*test_return_first.initial_rho
print(np.sum(assignments.theta - test_return_first.graphon*test_return_first.initial_rho))
print(assignments.theta[0])
print(scaled[0])

0.21007056686821174
[0.64427988 0.61621007 0.61633632 0.57618988 0.53743214 0.46016917
 0.26385557 0.22863275 0.1849514  0.19378866 0.17422043 0.19542987
 0.18217397 0.17194799 0.17598788 0.16323697 0.17043303 0.14063881
 0.13167529 0.12170181 0.12624669 0.13053907 0.13457897 0.1371039
 0.14253251 0.14101755 0.13382149 0.12687792 0.0724656  0.06956192
 0.06893069 0.06590077 0.06627951 0.0800404  0.08559525 0.09657871
 0.09796743 0.09165509 0.08875142 0.07688423 0.06716324 0.06577452
 0.06804696 0.06968817 0.0724656  0.07347557 0.07259184 0.07233935
 0.06766822 0.06590077 0.09102386 0.11488448 0.11463199 0.10238606
 0.11198081 0.10352228 0.12851124]
[0.6440069  0.61594899 0.61607518 0.57594575 0.53720444 0.4599742
 0.26374378 0.22853588 0.18487303 0.19370656 0.17414661 0.19534707
 0.18209678 0.17187513 0.17591332 0.1631678  0.17036081 0.14057922
 0.1316195  0.12165024 0.1261932  0.13048376 0.13452195 0.13704581
 0.14247212 0.1409578  0.13376479 0.12682416 0.07243489 0.06953245
 0.068901

In [9]:
smoothing_new = smooth_test._smoothing_histLC(test_return_first, A)

84


In [10]:
print(len(np.unique(smoothing_new[-1].get_graphon())))
print(len(smoothing_new))

84
84


In [11]:
smoothing_old,nclust_hist,nrow_tile = smoothing_histLC(test_return_first, A)
smoothed_old = flat_to_tensor(smoothing_old, nclust_hist, nrow_tile, test_return_first, n)

In [13]:
tensor_graphon = np.array(
            [smooth_test._from_graphon_to_adjacency(
                smoothing_new[i], assignments) for i in range(len(smoothing_new))]
        )

In [ ]:
tensor_P_new_2 = np.array(
    [g._get_edge_probabilities(n=n, exchangeable=False) for g in smoothing_new]
)

In [16]:
print(tensor_graphon.shape == smoothed_old.shape)
print(np.allclose(tensor_graphon, smoothed_old))
print(np.max(tensor_graphon))

True
False
0.7499444704841005


In [ ]:
error_diff_smoothing = [np.sum(tensor_P_new_2[i]- smoothed_old[i]) for i in range(len(smoothing_new))]
plt.plot(error_diff_smoothing)

# Testing if the tensor of smoothed graphon returned is the same

In [ ]:
tensor_new = smooth_test._flat_to_tensor(smoothing_new, test_return_first.get_graphon(), A.shape[0])
nclust = smoothing_old.shape[0]
nrow = test_return_first.get_graphon().graphon.shape[0]
tensor_old = flat_to_tensor(smoothing_old, nclust, nrow, test_return_first.get_graphon(), n)
np.sum(tensor_new - tensor_old)

In [ ]:
from SmoothNetHist import SmoothNetHist

my_smooth = SmoothNetHist(bandwidth=1/k)